# Text Translation and Sentiment Analysis using Transformers

## Project Overview:

The objective of this project is to analyze the sentiment of movie reviews in three different languages - English, French, and Spanish. We have been given 30 movies, 10 in each language, along with their reviews and synopses in separate CSV files named `movie_reviews_eng.csv`, `movie_reviews_fr.csv`, and `movie_reviews_sp.csv`.

- The first step of this project is to convert the French and Spanish reviews and synopses into English. This will allow us to analyze the sentiment of all reviews in the same language. We will be using pre-trained transformers from HuggingFace to achieve this task.

- Once the translations are complete, we will create a single dataframe that contains all the movies along with their reviews, synopses, and year of release in all three languages. This dataframe will be used to perform sentiment analysis on the reviews of each movie.

- Finally, we will use pretrained transformers from HuggingFace to analyze the sentiment of each review. The sentiment analysis results will be added to the dataframe. The final dataframe will have 30 rows


The output of the project will be a CSV file with a header row that includes column names such as **Title**, **Year**, **Synopsis**, **Review**, **Review Sentiment**, and **Original Language**. The **Original Language** column will indicate the language of the review and synopsis (*en/fr/sp*) before translation. The dataframe will consist of 30 rows, with each row corresponding to a movie.

In [1]:
# imports
import pandas as pd
import os
from transformers import MarianMTModel, MarianTokenizer
from transformers import pipeline

os.makedirs("result", exist_ok=True)

### Get data from `.csv` files and then preprocess data

In [2]:
# Read the 3 movie review CSV files with different column names in native language
df_eng = pd.read_csv("data/movie_reviews_eng.csv")
df_fr = pd.read_csv("data/movie_reviews_fr.csv").rename(
    columns={"Titre": "Movie / TV Series", "Année": "Year", "Critiques": "Review"}
)
df_sp = pd.read_csv("data/movie_reviews_sp.csv").rename(
    columns={"Título": "Movie / TV Series", "Año": "Year", "Sinopsis": "Synopsis", "Críticas": "Review"}
)

# Add an Original Language column so we can track source
df_eng["Original Language"] = "en"
df_fr["Original Language"] = "fr"
df_sp["Original Language"] = "es"

# Merge the three dataframes
df = pd.concat([df_eng, df_fr, df_sp], ignore_index=True)
df.shape

(30, 5)

In [3]:
df.sample(10)

,Movie / TV Series,Year,Synopsis,Review,Original Language
10,La La Land,2016,Cette comédie musicale raconte l'histoire d'un...,"""La La Land est un film absolument magnifique ...",fr
24,Amores perros,2000,Tres historias se entrelazan en esta película ...,"""Amores perros es una película intensa y conmo...",es
1,The Dark Knight,2008,Batman (Christian Bale) teams up with District...,"""The Dark Knight is a thrilling and intense su...",en
13,Les Choristes,2004,Ce film raconte l'histoire d'un professeur de ...,"""Les Choristes est un film magnifique qui vous...",fr
25,Águila Roja,(2009-2016),Esta serie de televisión española sigue las av...,"""Águila Roja es una serie aburrida y poco inte...",es
6,Scott Pilgrim vs. the World,2010,Scott Pilgrim (Michael Cera) must defeat his n...,"""It was difficult to sit through the whole thi...",en
20,Roma,2018,Cleo (Yalitza Aparicio) es una joven empleada ...,"""Roma es una película hermosa y conmovedora qu...",es
16,La Tour Montparnasse Infernale,2001,Deux employés de bureau incompétents se retrou...,"""Je ne peux pas croire que j'ai perdu du temps...",fr
14,Le Fabuleux Destin d'Amélie Poulain,2001,Cette comédie romantique raconte l'histoire d'...,"""Le Fabuleux Destin d'Amélie Poulain est un fi...",fr
4,Inception,2010,Dom Cobb (Leonardo DiCaprio) is a skilled thie...,"""Inception is a mind-bending and visually stun...",en


### Text translation

Translate the **Review** and **Synopsis** column values to English.

In [4]:
# load MarianMT translation models + tokenizers
fr_en_model_name = "Helsinki-NLP/opus-mt-fr-en"
es_en_model_name = "Helsinki-NLP/opus-mt-es-en"

fr_en_model = MarianMTModel.from_pretrained(fr_en_model_name)
fr_en_tokenizer = MarianTokenizer.from_pretrained(fr_en_model_name)

es_en_model = MarianMTModel.from_pretrained(es_en_model_name)
es_en_tokenizer = MarianTokenizer.from_pretrained(es_en_model_name)

print("loaded MarianMT fr-en and es-en")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

/opt/homebrew/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


loaded MarianMT fr-en and es-en


In [5]:
import torch


def translate_batch(texts, model, tokenizer):
    outputs = []
    for txt in texts:
        enc = tokenizer([txt], return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            gen = model.generate(**enc, max_length=512)
        outputs.append(tokenizer.decode(gen[0], skip_special_tokens=True))
    return outputs


# Translate French rows' Review + Synopsis
fr_mask = df["Original Language"] == "fr"
df.loc[fr_mask, "Review"] = translate_batch(df.loc[fr_mask, "Review"].tolist(), fr_en_model, fr_en_tokenizer)
df.loc[fr_mask, "Synopsis"] = translate_batch(df.loc[fr_mask, "Synopsis"].tolist(), fr_en_model, fr_en_tokenizer)

# Translate Spanish rows
es_mask = df["Original Language"] == "es"
df.loc[es_mask, "Review"] = translate_batch(df.loc[es_mask, "Review"].tolist(), es_en_model, es_en_tokenizer)
df.loc[es_mask, "Synopsis"] = translate_batch(df.loc[es_mask, "Synopsis"].tolist(), es_en_model, es_en_tokenizer)

print("translation done")

translation done


In [6]:
df.sample(10)

,Movie / TV Series,Year,Synopsis,Review,Original Language
9,The Island,2005,In a future where people are cloned for organ ...,"""The Island is a bland and forgettable sci-fi ...",en
27,El Bar,2017,A group of people are trapped in a bar after M...,"""The Bar is a ridiculous and meaningless film ...",es
8,Solo: A Star Wars Story,2018,A young Han Solo (Alden Ehrenreich) joins a gr...,"""Dull and pointless, with none of the magic of...",en
11,Intouchables,2011,This film tells the story of the unlikely frie...,"""Untouchables is an incredibly touching film w...",fr
2,Forrest Gump,1994,Forrest Gump (Tom Hanks) is a simple man with ...,"""Forrest Gump is a heartwarming and inspiratio...",en
7,The Nice Guys,2016,"In 1970s Los Angeles, a private eye (Ryan Gosl...","""The Nice Guys tries too hard to be funny, and...",en
12,Amélie,2001,This romantic comedy tells the story of Amélie...,"""Amélie is an absolutely charming film that wi...",fr
18,Les Visiteurs en Amérique,2000,In this continuation of the French comedy The ...,"""The film is a total waste of time. The jokes ...",fr
23,El Laberinto del Fauno,2006,"During the Spanish postwar period, Ofelia (Iva...","""The Labyrinth of Fauno is a fascinating and e...",es
20,Roma,2018,Cleo (Yalitza Aparicio) is a young domestic wo...,"""Rome is a beautiful and moving film that pays...",es


### Sentiment Analysis

Use HuggingFace pretrained model for sentiment analysis of the reviews. Store the sentiment result **Positive** or **Negative** in a new column titled **Sentiment** in the dataframe.

In [7]:
# Load sentiment analysis pipeline
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_classifier = pipeline("sentiment-analysis", model=model_name)


def analyze_sentiment(review: str) -> str:
    """Return 'Positive' or 'Negative' for a given review text."""
    result = sentiment_classifier(review[:512])[0]
    return result["label"].capitalize()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [8]:
# Apply sentiment analysis to every review and store as new column
df["Sentiment"] = df["Review"].apply(analyze_sentiment)
df["Sentiment"].value_counts()

Sentiment
Positive    15
Negative    15
Name: count, dtype: int64

In [9]:
df.sample(10)

,Movie / TV Series,Year,Synopsis,Review,Original Language,Sentiment
0,The Shawshank Redemption,1994,"Andy Dufresne (Tim Robbins), a successful bank...","""The Shawshank Redemption is an inspiring tale...",en,Positive
13,Les Choristes,2004,This film tells the story of a music teacher w...,"""The Choristes are a beautiful film that will ...",fr,Positive
16,La Tour Montparnasse Infernale,2001,Two incompetent office workers find themselves...,"""I can't believe I've wasted time watching thi...",fr,Negative
23,El Laberinto del Fauno,2006,"During the Spanish postwar period, Ofelia (Iva...","""The Labyrinth of Fauno is a fascinating and e...",es,Positive
22,Y tu mamá también,2001,Two teenage friends (Gael García Bernal and Di...,"""And your mom is also a movie that stays with ...",es,Positive
2,Forrest Gump,1994,Forrest Gump (Tom Hanks) is a simple man with ...,"""Forrest Gump is a heartwarming and inspiratio...",en,Positive
17,Astérix aux Jeux Olympiques,2008,In this film adaptation of the popular comic s...,"""This film is a complete disappointment. The j...",fr,Negative
1,The Dark Knight,2008,Batman (Christian Bale) teams up with District...,"""The Dark Knight is a thrilling and intense su...",en,Positive
15,Le Dîner de Cons,1998,The film follows the story of a group of rich ...,"""I didn't like this movie at all. The concept ...",fr,Negative
21,La Casa de Papel,(2017-2021),This Spanish television series follows a group...,"""The Paper House is an exciting and addictive ...",es,Positive


In [10]:
# export the results to a .csv file
df.to_csv("result/reviews_with_sentiment.csv", index=False)